# 第17课：预训练语言模型——BERT 与 GPT

## 学习目标
- 理解「预训练 + 微调」范式的核心思想
- 区分 BERT（双向编码器）和 GPT（单向解码器）的设计哲学
- 用代码直观理解 Masked Language Model 和 Causal Language Model 的训练目标
- 掌握这两个模型如何引爆了 NLP 和大模型的革命

## 核心概念

上一课我们学了 Transformer 架构——Self-Attention、多头注意力、位置编码。

Transformer 本身只是一个「计算框架」。真正改变世界的是两个基于 Transformer 的预训练语言模型：

- **BERT（2018）**：用 Transformer Encoder，双向看上下文，擅长「理解」
- **GPT（2018-2020）**：用 Transformer Decoder，从左到右生成，擅长「生成」

它们的共同革命性在于：**先在海量文本上无监督预训练，再在下游任务上微调**。
这意味着不再需要为每个任务从零标注大量数据。

```
Transformer → BERT（理解派）/ GPT（生成派）→ 预训练范式 → 大模型时代
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. 预训练范式的直觉

传统方式：为每个任务标注数据 → 从零训练模型 → 效果取决于数据量

预训练方式：先让模型「读」海量文本（学会语言规律）→ 再用少量标注数据微调

**类比：** 传统方式像让一个人从零学医；预训练方式像先让他读完医学百科，再实习。

In [ ]:
# 可视化：预训练 vs 传统方式的效果对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：数据量 vs 效果
data_sizes = [100, 500, 1000, 5000, 10000, 50000]
traditional = [45, 55, 62, 70, 75, 78]
pretrained = [65, 72, 78, 84, 88, 91]

axes[0].plot(data_sizes, traditional, 'o-', label='从零训练', color='#C96442')
axes[0].plot(data_sizes, pretrained, 's-', label='预训练+微调', color='#4A90D9')
axes[0].set_xlabel('标注数据量')
axes[0].set_ylabel('准确率 (%)')
axes[0].set_title('预训练范式的优势')
axes[0].legend()
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)

# 右图：BERT vs GPT 架构对比
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')

# BERT: 双向
bert_box = mpatches.FancyBboxPatch((0.5, 1), 3.5, 8, boxstyle="round,pad=0.3", 
                                     facecolor='#E8D5C4', edgecolor='#C96442', linewidth=2)
ax.add_patch(bert_box)
ax.text(2.25, 8.5, 'BERT', fontsize=14, fontweight='bold', ha='center', color='#C96442')
ax.text(2.25, 7, 'Transformer\nEncoder', ha='center', fontsize=10)
ax.annotate('', xy=(0.8, 5.5), xytext=(3.7, 5.5),
            arrowprops=dict(arrowstyle='<->', color='#C96442', lw=2))
ax.text(2.25, 5, '双向注意力', ha='center', fontsize=9, color='#C96442')
ax.text(2.25, 3.5, 'MLM 目标\n完形填空', ha='center', fontsize=10)
ax.text(2.25, 1.8, '擅长: 理解\n分类/问答/NER', ha='center', fontsize=9)

# GPT: 单向
gpt_box = mpatches.FancyBboxPatch((5.5, 1), 3.5, 8, boxstyle="round,pad=0.3",
                                    facecolor='#D4E6F1', edgecolor='#4A90D9', linewidth=2)
ax.add_patch(gpt_box)
ax.text(7.25, 8.5, 'GPT', fontsize=14, fontweight='bold', ha='center', color='#4A90D9')
ax.text(7.25, 7, 'Transformer\nDecoder', ha='center', fontsize=10)
ax.annotate('', xy=(5.8, 5.5), xytext=(8.7, 5.5),
            arrowprops=dict(arrowstyle='->', color='#4A90D9', lw=2))
ax.text(7.25, 5, '单向注意力\n(只看左边)', ha='center', fontsize=9, color='#4A90D9')
ax.text(7.25, 3.5, 'CLM 目标\n预测下一个', ha='center', fontsize=10)
ax.text(7.25, 1.8, '擅长: 生成\n对话/续写/创作', ha='center', fontsize=9)

ax.axis('off')
ax.set_title('BERT vs GPT 架构对比')
plt.tight_layout()
plt.savefig('bert_vs_gpt.png', dpi=100, bbox_inches='tight')
plt.show()
print('架构对比图已生成')

In [ ]:
# BERT 的 Masked Language Model (MLM) 直觉演示
# 完形填空：随机遮住 15% 的词，让模型根据上下文猜

import random
random.seed(42)

sentence = "机器学习 是 人工智能 的 一个 重要 分支".split()

# 随机 mask 15% 的 token
mask_ratio = 0.15
n_mask = max(1, int(len(sentence) * mask_ratio))
mask_indices = sorted(random.sample(range(len(sentence)), n_mask))

masked = sentence.copy()
original = {}
for idx in mask_indices:
    original[idx] = masked[idx]
    masked[idx] = '[MASK]'

print('='*60)
print('BERT 训练目标: Masked Language Model (完形填空)')
print('='*60)
print(f'原始句子:  {" ".join(sentence)}')
print(f'遮盖后:    {" ".join(masked)}')
print(f'模型需要预测: { {idx: original[idx] for idx in mask_indices} }')
print()
print('关键: BERT 可以同时看到 [MASK] 左边和右边的上下文 → 双向理解')
print('这就是为什么 BERT 在理解任务上很强')

# 简化版 MLM 概率模拟
# 假设词表只有这些词，模型对被 mask 位置的预测概率
vocab = sentence + ['技术', '方法', '领域', '工具']
print(f'\n--- 对位置 {mask_indices[0]} 的预测分布（模拟） ---')
probs = {w: round(random.random(), 3) for w in vocab}
total = sum(probs.values())
probs = {w: round(p/total, 3) for w, p in sorted(probs.items(), key=lambda x: -x[1])}
for w, p in list(probs.items())[:5]:
    bar = '█' * int(p * 50)
    print(f'  {w:8s} {p:.3f} {bar}')

In [ ]:
# GPT 的 Causal Language Model (CLM) 直觉演示
# 自回归：每次只根据左边已生成的词，预测下一个词

print('='*60)
print('GPT 训练目标: Causal Language Model (预测下一个词)')
print('='*60)

tokens = "深度 学习 改变了 自然 语言 处理 的 方向".split()
print(f'完整句子: {" ".join(tokens)}')
print()

# 展示自回归过程
next_word_probs = {
    '深度': {'学习': 0.72, '理解': 0.10, '思考': 0.08, '分析': 0.06, '训练': 0.04},
    '深度 学习': {'改变': 0.45, '推动': 0.25, '是': 0.15, '在': 0.10, '让': 0.05},
    '深度 学习 改变': {'了': 0.80, '世界': 0.08, '人类': 0.07, '一切': 0.05},
}

for context, probs in next_word_probs.items():
    print(f'输入: "{context}" → 预测下一个词:')
    for word, p in probs.items():
        bar = '█' * int(p * 30)
        print(f'  {word:4s} {p:.2f} {bar}')
    print()

print('关键: GPT 只能看到左边已生成的词 → 自回归生成')
print('这就是为什么 GPT 擅长生成连贯的文本')
print()
print('GPT-1(2018) → GPT-2(2019) → GPT-3(2020): 核心架构不变，规模不断增大')
print('GPT-3: 1750亿参数，少样本学习(in-context learning)的开端')

In [ ]:
# BERT vs GPT 关键参数演进
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))

models = ['BERT-Base\n(2018)', 'BERT-Large\n(2018)', 'GPT-1\n(2018)', 
          'GPT-2\n(2019)', 'GPT-3\n(2020)']
params_m = [110, 340, 117, 1500, 175000]  # 百万参数
colors = ['#C96442', '#C96442', '#4A90D9', '#4A90D9', '#4A90D9']

bars = ax.bar(models, [p/1000 for p in params_m], color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylabel('参数量（十亿）', fontsize=12)
ax.set_title('预训练语言模型的参数量增长', fontsize=14)
ax.set_yscale('log')

# 添加数值标签
for bar, p in zip(bars, params_m):
    label = f'{p/1000:.1f}B' if p >= 1000 else f'{p}M'
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.2,
            label, ha='center', va='bottom', fontweight='bold')

# 图例
import matplotlib.patches as mpatches
legend_elements = [mpatches.Patch(facecolor='#C96442', label='BERT (Encoder)'),
                   mpatches.Patch(facecolor='#4A90D9', label='GPT (Decoder)')]
ax.legend(handles=legend_elements, loc='upper left')

ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 打印对比表
print('\n' + '='*70)
print('BERT vs GPT 核心对比')
print('='*70)
print(f'{"维度":<15} {"BERT":<25} {"GPT":<25}')
print('-'*65)
comparisons = [
    ('架构', 'Transformer Encoder', 'Transformer Decoder'),
    ('注意力', '双向（看左右）', '单向（只看左边）'),
    ('训练目标', 'MLM（完形填空）', 'CLM（预测下一个）'),
    ('预训练数据', 'BooksCorpus + Wiki', 'WebText / CommonCrawl'),
    ('擅长任务', '分类/问答/NER', '生成/对话/续写'),
    ('代表突破', '11项NLP基准SOTA', 'GPT-3 少样本学习'),
    ('后续影响', '开启了NLP预训练时代', '开启了生成式AI时代'),
]
for dim, bert, gpt in comparisons:
    print(f'{dim:<13} {bert:<25} {gpt:<25}')

## 总结

| 要点 | 内容 |
|------|------|
| 预训练范式 | 先在海量文本上无监督学习语言规律，再用少量标注数据微调 |
| BERT | 双向 Encoder，MLM 目标，擅长理解任务（分类、问答） |
| GPT | 单向 Decoder，CLM 目标，擅长生成任务（对话、创作） |
| 规模法则 | 参数量从亿级到千亿级，效果持续提升 |
| 历史意义 | 这两个模型开启了「预训练语言模型」时代，是大模型的直接前身 |

## 课后思考

1. **为什么 BERT 用 MLM 而不是 CLM？** 双向注意力让模型能看到完整上下文，对理解任务更有效。但 MLM 不适合生成（不能在推理时 mask）。

2. **GPT 从 1 到 3，架构几乎没变，为什么效果差距这么大？** 核心是规模——数据量、参数量、计算量。这就是后来被总结为「Scaling Law」的发现。

3. **为什么最终是 GPT 路线胜出？** 生成能力 + 规模扩展 + In-Context Learning 的涌现，使得 GPT 路线成为了 ChatGPT 等 AI 助手的基础。